# Bengaluru Survey Data Analysis

This notebook documents the data quality checks, cleaning, validation, summary analysis, and visualizations for the Bengaluru ward-level survey dataset.

## 1. Load Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("../data/Bengaluru_Survey_Public_Ward_Analysis.xlsx")
df = pd.read_excel(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

## 2. Data Quality Check

In [ ]:
print("Missing values:")
print(df.isnull().sum())

## 3. Data Cleaning

The source data has one missing `Commercial` value for Ward 161 (Hosakerehalli). The existing analysis logic records this value as 1239.

In [ ]:
clean_df = df.copy()

clean_df.loc[
    (clean_df["Ward Number"] == 161) &
    (clean_df["Ward Name"] == "Hosakerehalli"),
    "Commercial"
] = 1239

print("Commercial value for Ward 161:",
      clean_df.loc[clean_df["Ward Number"] == 161, "Commercial"].iloc[0])

## 4. Validate Balance Left

In [ ]:
clean_df["Calculated Balance"] = (
    clean_df["Total Building Count"] -
    clean_df["Verification Done"]
)

balance_errors = clean_df[
    clean_df["Balance Left"] != clean_df["Calculated Balance"]
]

print("Balance validation errors:", len(balance_errors))

## 5. Validate Verification Percentage

In [ ]:
clean_df["Calculated % Done"] = (
    clean_df["Verification Done"] /
    clean_df["Total Building Count"] * 100
)

percentage_errors = clean_df[
    abs(clean_df["% Done"] - clean_df["Calculated % Done"]) > 0.01
]

print("Percentage validation errors:", len(percentage_errors))

## 6. Overall Verification Performance

In [ ]:
overall_verification = (
    clean_df["Verification Done"].sum() /
    clean_df["Total Building Count"].sum() * 100
)

print("Overall verification percentage:",
      round(overall_verification, 2), "%")

## 7. Ward Count by Corporation

In [ ]:
ward_count = clean_df.groupby("Corporation")["Ward Number"].count()
print(ward_count)

## 8. Recorded Positive Balance by Division

In [ ]:
division_backlog = (
    clean_df.groupby("Division")["Balance Left"]
    .sum()
    .sort_values(ascending=False)
)

positive_backlog = division_backlog[division_backlog > 0]
print(positive_backlog)

## 9. Top Wards by Recorded Positive Balance

In [ ]:
top_backlog_wards = (
    clean_df[clean_df["Balance Left"] > 0]
    .sort_values("Balance Left", ascending=False)
)

print(top_backlog_wards[
    ["Ward Number", "Ward Name", "Division", "Balance Left"]
].head(10))

## 10. Property Type Totals

In [ ]:
property_totals = clean_df[
    ["Residential Households", "Commercial", "Apartments"]
].sum()

print(property_totals)

## 11. Visualizations

In [ ]:
# Top 10 wards
top = clean_df[clean_df["Balance Left"] > 0].sort_values(
    "Balance Left", ascending=False
).head(10).sort_values("Balance Left")

plt.figure(figsize=(10,6))
plt.barh(top["Ward Name"], top["Balance Left"])
plt.xlabel("Recorded Balance Left")
plt.ylabel("Ward")
plt.title("Top 10 Wards by Recorded Positive Balance")
plt.tight_layout()
plt.show()

# Property totals
plt.figure(figsize=(8,5))
pt = clean_df[
    ["Residential Households", "Commercial", "Apartments"]
].sum()
plt.bar(pt.index, pt.values)
plt.ylabel("Count")
plt.title("Property Type Totals")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 12. Final Dataset

Temporary validation columns are removed before the final analysis dataframe is retained.

In [ ]:
clean_df = clean_df.drop(
    columns=["Calculated Balance", "Calculated % Done"]
)

print("Final cleaned dataset shape:", clean_df.shape)

## Key Findings

- The dataset contains 198 ward records and 11 source columns.
- Balance and verification-percentage calculations validate against the source values within the checks used above.
- The overall verification percentage is calculated from total verification done divided by total building count.
- Ward-level and division-level recorded positive balances are summarized to identify where remaining recorded balance is concentrated.
- Property totals are calculated after applying the documented Ward 161 Commercial correction.